# 02 — Extract USGS Daily Values

This notebook pulls USGS daily values for the top California sites found in the WQP extract.
Default predictors:
- discharge (`00060`)
- water temperature (`00010`)
- gage height (`00065`)

In [ ]:
import sys
from pathlib import Path
import yaml
import pandas as pd

project_root = Path("..").resolve()
sys.path.append(str(project_root / "src"))

from usgs import batch_fetch_usgs_daily, aggregate_usgs_daily

In [ ]:
cfg = yaml.safe_load(open(project_root / "config" / "config.sample.yaml"))
wqp_wide = pd.read_csv(project_root / "data" / "interim" / "wqp_wide.csv")
site_counts = (
    wqp_wide.dropna(subset=["usgs_site_no"])
    .groupby("usgs_site_no")
    .size()
    .reset_index(name="n_rows")
    .sort_values("n_rows", ascending=False)
)
site_counts.head()

In [ ]:
top_sites = site_counts["usgs_site_no"].head(40).tolist()
usgs_long = batch_fetch_usgs_daily(
    site_list=top_sites,
    parameter_map=cfg["usgs"]["parameters"],
    start_date=cfg["project"]["start_date"],
    end_date=cfg["project"]["end_date"],
)
usgs_long.head()

In [ ]:
usgs_wide = aggregate_usgs_daily(usgs_long)
print("USGS long rows:", len(usgs_long))
print("USGS wide rows:", len(usgs_wide))
usgs_wide.head()

In [ ]:
usgs_long.to_csv(project_root / "data" / "raw" / "usgs_long.csv", index=False)
usgs_wide.to_csv(project_root / "data" / "interim" / "usgs_wide.csv", index=False)